# HTR CREMMA Medieval — Expérience 2 : Arrow grayscale (fix mismatch L/1)

**Hypothèse testée :** le modèle de base `cremma-generic-1.0.1` est entraîné sur des images
grayscale (mode L). Toutes les runs précédentes utilisaient des données binarisées (mode 1),
créant un mismatch qui plafonnait le CER à ~26%. Cette run utilise les Arrow compilés depuis
`preprocessed_grayscale/` (mode L confirmé).

**Données :**
- `train.arrow` — 939 MB — 19 797 lignes — mode L
- `dev.arrow` — 144 MB — 3 729 lignes — mode L

**Avant de lancer :**
1. `Exécution` → `Modifier le type d'exécution` → **GPU A100** (ou T4)
2. Configurer `AWS_ACCESS_KEY_ID` et `AWS_SECRET_ACCESS_KEY` dans les Secrets Colab (icône cadenas)

**Ordre d'exécution :** 0 → 1 → 2 → 6a → 6b → 7

In [ ]:
# Cellule 0 — Vérifier le GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'Aucun GPU — activer dans Exécution > Modifier le type d\'exécution')

In [ ]:
# Cellule 1 — Clés AWS via Colab Secrets
import os

try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    os.environ['AWS_DEFAULT_REGION']    = 'eu-west-3'
    print('Cles chargees depuis Colab Secrets')
except Exception as e:
    print(f'ERREUR : {e}')
    print('Configurer AWS_ACCESS_KEY_ID et AWS_SECRET_ACCESS_KEY dans les Secrets Colab')

In [ ]:
# Cellule 2 — Installer Kraken
import subprocess, sys, torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU : {torch.cuda.get_device_name(0)} (sm_{cap[0]}{cap[1]})')
    print('bf16 supporte' if cap[0] >= 8 else 'fp16 uniquement')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kraken', 'iso639-lang', 'opencv-python-headless', 'lxml', 'tqdm', 'boto3',
], check=True)
print('Kraken installe')

test = subprocess.run(
    [sys.executable, '-c',
     'from kraken.lib.xml import XMLPage; '
     'from kraken.train import KrakenTrainer; '
     'print("Imports OK")'],
    capture_output=True, text=True
)
print(test.stdout.strip())
if test.returncode != 0:
    print('ERREUR :', test.stderr[-1500:])

result = subprocess.run(['ketos', '--version'], capture_output=True, text=True)
print(f'ketos : {result.stdout.strip() or result.stderr.strip()}')

In [ ]:
# Cellule 6a — Telecharger les Arrow grayscale bruts + modele de base depuis S3
import os, boto3, time

s3 = boto3.client('s3', region_name='eu-west-3')
BUCKET = 'htr-cremma-medieval'

os.makedirs('/content/splits', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

for s3_key, local in [
    ('splits/train.arrow',                      '/content/splits/train.arrow'),
    ('splits/dev.arrow',                        '/content/splits/dev.arrow'),
    ('base-model/cremma-generic-1.0.1.mlmodel', '/content/cremma_generic.mlmodel'),
]:
    print(f'Telechargement {s3_key}...')
    t0 = time.time()
    s3.download_file(BUCKET, s3_key, local)
    size = os.path.getsize(local) / 1024 / 1024
    print(f'OK : {local} ({size:.1f} MB | {time.time()-t0:.0f}s)')

with open('/content/splits/train_binary.txt', 'w') as f:
    f.write('/content/splits/train.arrow\n')
with open('/content/splits/dev_binary.txt', 'w') as f:
    f.write('/content/splits/dev.arrow\n')

print('Manifests OK')
print('train.arrow : 19 797 lignes — mode L (grayscale)')
print('dev.arrow   : 3 729 lignes  — mode L (grayscale)')

In [ ]:
# Cellule 6b — Fine-tuning Exp 2 (Arrow grayscale brut)
# Auto-detecte A100 (bf16, batch=16) vs T4 (fp16, batch=8)
import subprocess, os, time, datetime, re, torch

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
IS_AMPERE = cap[0] >= 8

PRECISION = 'bf16-mixed' if IS_AMPERE else '16-mixed'
BATCH     = 16 if IS_AMPERE else 8
WORKERS   = 11 if IS_AMPERE else 4
print(f'GPU : {GPU_NAME} | precision={PRECISION} | batch={BATCH} | workers={WORKERS}')

cmd = [
    'ketos',
    '--device', 'cuda:0',
    '--workers', str(WORKERS),
    '--precision', PRECISION,
    '--threads', '4',
    'train',
    '-f', 'binary',
    '-t', '/content/splits/train_binary.txt',
    '-e', '/content/splits/dev_binary.txt',
    '-i', '/content/cremma_generic.mlmodel',
    '--resize', 'union',
    '-o', '/content/models/htr_cremma_exp2',
    '-q', 'early',
    '-N', '50',
    '--lag', '10',
    '--min-epochs', '3',
    '-B', str(BATCH),
    '-r', '0.0001',
    '--augment',
]

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TERM'] = 'dumb'

print('Commande :', ' '.join(cmd))
print(f'Debut    : {datetime.datetime.now().strftime("%H:%M:%S")}')
print('-' * 60)

t_global  = time.time()
t_epoch   = time.time()
epoch_num = 0
best_acc  = 0.0
best_epoch = 0

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=0, env=env)
buf = ''
while True:
    ch = proc.stdout.read(1)
    if not ch:
        break
    if ch == '\r':
        print('\r' + buf, end='', flush=True)
        buf = ''
    elif ch == '\n':
        line = buf
        print(line)

        m = re.search(r'stage\s+(\d+)', line, re.IGNORECASE)
        if m:
            epoch_num = int(m.group(1))
            t_epoch = time.time()

        m_acc = re.search(r'val_accuracy[:\s]+([\d.]+)', line, re.IGNORECASE)
        if m_acc:
            acc = float(m_acc.group(1))
            elapsed_epoch = time.time() - t_epoch
            elapsed_total = time.time() - t_global
            if acc > best_acc:
                best_acc = acc
                best_epoch = epoch_num
            print(f'  [Epoch {epoch_num:>2}] acc={acc*100:.2f}% | CER={((1-acc)*100):.2f}% | '
                  f'{elapsed_epoch:.0f}s | total={elapsed_total/60:.1f}min')
            print(f'  Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% / CER {(1-best_acc)*100:.2f}%')

        m_pat = re.search(r'patience\s+(\d+)/(\d+)', line, re.IGNORECASE)
        if m_pat:
            cur, total_p = int(m_pat.group(1)), int(m_pat.group(2))
            print(f'  Patience : {cur}/{total_p} — stop dans {total_p - cur} epoch(s)')

        buf = ''
    else:
        buf += ch

if buf:
    print(buf)
proc.wait()

elapsed_total = time.time() - t_global
print('-' * 60)
print(f'Fin : {datetime.datetime.now().strftime("%H:%M:%S")} | Duree : {elapsed_total/60:.1f} min')
print(f'Meilleur : epoch {best_epoch} -> {best_acc*100:.2f}% acc / CER {(1-best_acc)*100:.2f}%')
print(f'Code retour : {proc.returncode}')

In [ ]:
# Cellule 7 — Uploader le modele final sur S3
import glob, datetime
from pathlib import Path

models_dir = '/content/models'
timestamp  = datetime.datetime.now().strftime('%Y%m%d_%H%M')

mlmodels    = glob.glob(f'{models_dir}/**/*.mlmodel', recursive=True)
safetensors = glob.glob(f'{models_dir}/**/*.safetensors', recursive=True)
checkpoints = glob.glob(f'{models_dir}/**/*.ckpt', recursive=True)

print('Fichiers trouves :')
print(f'  .mlmodel     : {mlmodels}')
print(f'  .safetensors : {safetensors}')
print(f'  .ckpt        : {checkpoints}')

uploaded = []

for f in mlmodels:
    nom = f'exp2_grayscale_{timestamp}.mlmodel'
    s3.upload_file(f, BUCKET, f'output/{nom}')
    print(f'Upload : s3://{BUCKET}/output/{nom}')
    uploaded.append(nom)

if not mlmodels:
    for f in safetensors:
        nom = f'exp2_grayscale_{timestamp}.safetensors'
        s3.upload_file(f, BUCKET, f'output/{nom}')
        print(f'Upload : s3://{BUCKET}/output/{nom}')
        uploaded.append(nom)

if not mlmodels and not safetensors and checkpoints:
    best = sorted(checkpoints, key=lambda p: float(Path(p).stem.split('-')[-1]) if '-' in Path(p).stem else 0, reverse=True)[0]
    nom = f'exp2_grayscale_{timestamp}.ckpt'
    s3.upload_file(best, BUCKET, f'output/{nom}')
    print(f'Upload (checkpoint) : s3://{BUCKET}/output/{nom}')
    uploaded.append(nom)

if not uploaded:
    print('ERREUR : aucun modele trouve dans', models_dir)
    for f in Path(models_dir).rglob('*'):
        print(f'  {f}')
else:
    print(f'OK — {len(uploaded)} fichier(s) uploade(s) sur S3')